# Atari Pong with frame preprocessing

This example uses standard [Gymnasium Atari (ALE)](https://gymnasium.farama.org/environments/atari/) envs — mouse-env does not change the games and has no Atari-specific code. Atari is configured like any other custom Gymnasium setup: build the env in an `env_fn` factory (`gym.make` + `gymnasium.wrappers.AtariPreprocessing`) and register ALE yourself with `gymnasium.register_envs(ale_py)`.

mouse-env passes through the preprocessed frame as `outputs[i]["observation"]`. After `AtariPreprocessing`, Pong emits an 84×84 `uint8` frame, and mouse-env preserves that shape and dtype.

## Install extra

Requires `pip install 'mouse-env[atari]'`, which installs `ale_py` (ALE ROMs) and `opencv-python` (needed by `AtariPreprocessing` for frame resizing).

In [ ]:
import ale_py
import gymnasium as gym
import numpy as np
from gymnasium.wrappers import AtariPreprocessing

from mouse_envs import EnvConfig, make_env, make_group_env

gym.register_envs(ale_py)

## Configuration

`frameskip=1` in the factory disables the base env's built-in frame skipping so `AtariPreprocessing` owns it entirely — a common pattern when using `AtariPreprocessing`.

Atari games already contain repeated internal structure, such as points, lives, and rounds. This example sets `episodes_per_task=1`, so each native Gymnasium/ALE game is one mouse-env task. You can raise it later if you want a task to span multiple full games.

In [ ]:
def make_pong():
    env = gym.make("ALE/Pong-v5", frameskip=1, max_episode_steps=10000)
    return AtariPreprocessing(env, frame_skip=4, screen_size=84, noop_max=30)


cfgs = [
    EnvConfig(
        id="ALE/Pong-v5",
        reset_seed=i,
        episodes_per_task=1,
        env_fn=make_pong,
    )
    for i in range(4)
]
env = make_group_env(cfgs)

## First step and observation shape

After preprocessing, each env emits an 84×84 frame in its native shape and dtype under the standard `observation` key.

In [ ]:
outputs = env.step(env.sample_random_input())

print(f"obs spec:  {env.output_specs[0].observation}")
batch_obs = np.stack([r["observation"].numpy() for r in outputs])
print(f"obs shape: {batch_obs.shape}")  # (env.num_envs, 84, 84)

## Show a preprocessed frame

Each observation is already an 84×84 grayscale frame — exactly what the agent sees after `AtariPreprocessing`.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, env.num_envs, figsize=(3 * env.num_envs, 3))
for ax, frame in zip(np.atleast_1d(axes), batch_obs):
    ax.imshow(frame, cmap="gray", vmin=0, vmax=255)
    ax.axis("off")
fig.tight_layout()
plt.show()

## Short follow-up rollout

In [ ]:
for _ in range(10):
    env.step(env.sample_random_input())

env.close()